# Tolerance Range Selection
## Context
To calculate the travel time between each stop for every bus trip, we perform a complex asof_join, which requires a manual setting of `tolerance` range to work well.
## Goal
Find the ideal tolerance for each stop pair for each route.

## (below for future documentation)
## Find ideal tolerance in asof_join
1. Required data: `bus_event_time.parquet`, `target_stop.json`
2. Create ideal_tolerances = []
2. For each route, set tolerance = '1h'
3. Perform asof_join, get the number of rows of resulting joined df (`nrows: int`)
3. Repeat the following
    - increase tolerance by '1h' (new_tolerance)
    - perform asof_join again, get new_nrows
    - if new_nrows / nrows < 1.03 or tolerance == "12h", ideal_tolerance = tolerance, break the loop
    - else, tolerance = new_tolerance, continue the loop
6. ideal_tolerances.append({"depart": int, "dest": int, "tolerance": str})
7. Create pl.DataFrame(ideal_tolerances)
8. df.write_csv

# Import

In [1]:
import polars as pl
from polars import Series
from constants import DATA_FILE, PROCESSED_DATA_FOLDER
from helpers import clean_df
import json
import plotly.express as px
from datetime import datetime, date, time, timedelta

In [2]:
with open(PROCESSED_DATA_FOLDER / "target_routes.json", "r", encoding="utf-8") as f:
    target_routes = json.load(f)
df_train = (
    pl.scan_parquet(DATA_FILE)
    .filter(pl.col("SubRouteID").is_in(target_routes))
    .pipe(clean_df)
)
with open(PROCESSED_DATA_FOLDER / "target_stops.json", "r", encoding="utf-8") as f:
    target_stops = json.load(f)
with open(PROCESSED_DATA_FOLDER / "stops_dict.json", "r", encoding="utf-8") as f:
    stops_dict = json.load(f)

In [4]:
def asof_join_by_id(
    route_df: pl.DataFrame,
    depart_id: int,
    dest_id: int,
    tolerance: str,
) -> int:
    depart_df = (
        route_df.filter(
            pl.col("Event") == "離站",
            pl.col("StopID") == depart_id,
        )
        .with_columns(pl.col("Time").alias("DepartureTime"))
        .sort("Time")
    )
    dest_df = (
        route_df.filter(
            pl.col("Event") == "進站",
            pl.col("StopID") == dest_id,
        )
        .with_columns(pl.col("Time").alias("ArrivalTime"))
        .sort("Time")
    )
    result = depart_df.join_asof(
        dest_df,
        on="Time",
        by="Plate",
        strategy="forward",
        tolerance=tolerance,
        check_sortedness=False,
    ).drop_nulls(subset=["ArrivalTime"])
    return len(result)

In [5]:
def find_ideal_tolerances(
    df: pl.LazyFrame,
    target_stops: dict[str, list[int]],
) -> pl.DataFrame:
    """
    Finds the ideal asof_join tolerance for each sequential stop pair per route.

    Args:
        df: Cleaned LazyFrame (output of clean_df).
        target_stops: Maps Route (str) to an ordered list of StopIDs (int).
                      Pairs are derived from consecutive stops in the list.

    Returns:
        A DataFrame with columns: route, depart_id, dest_id, tolerance.
    """

    def _find_tolerance_for_pair(
        route_df: pl.DataFrame,
        depart_id: int,
        dest_id: int,
    ) -> tuple[str, int]:
        MAX_HOURS = 12
        tolerance_hours = 1
        n_rows = asof_join_by_id(route_df, depart_id, dest_id, f"{tolerance_hours}h")

        while tolerance_hours < MAX_HOURS:
            new_tolerance_hours = tolerance_hours + 1
            new_nrows = asof_join_by_id(
                route_df, depart_id, dest_id, f"{new_tolerance_hours}h"
            )
            if n_rows > 0 and new_nrows / n_rows < 1.03:
                break
            tolerance_hours = new_tolerance_hours
            n_rows = new_nrows

        return (f"{tolerance_hours}h", n_rows)

    collected_df = df.select(["Route", "Plate", "StopID", "Event", "Time"]).collect(
        engine="streaming"
    )
    results: list[dict] = []
    count = 0
    for route, stop_ids in target_stops.items():
        route_df = collected_df.filter(pl.col("Route") == route)
        if route_df.is_empty():
            continue

        for depart_id, dest_id in zip(stop_ids, stop_ids[1:]):
            tolerance, n_rows = _find_tolerance_for_pair(route_df, depart_id, dest_id)
            results.append(
                {
                    "route": route,
                    "depart_id": depart_id,
                    "dest_id": dest_id,
                    "tolerance": tolerance,
                    "n_rows": n_rows,
                }
            )
        count += 1
        if count % 100 == 0:
            print(f"{count} routes have been processed.")

    return pl.DataFrame(
        results,
        schema={
            "route": pl.String,
            "depart_id": pl.Int32,
            "dest_id": pl.Int32,
            "tolerance": pl.String,
            "n_rows": pl.UInt16,
        },
    )

In [6]:
ideal_tolerances = find_ideal_tolerances(df_train, target_stops)

100 routes have been processed.
200 routes have been processed.
300 routes have been processed.
400 routes have been processed.
500 routes have been processed.
600 routes have been processed.
700 routes have been processed.
800 routes have been processed.
900 routes have been processed.
1000 routes have been processed.
1100 routes have been processed.
1200 routes have been processed.
1300 routes have been processed.
1400 routes have been processed.
1500 routes have been processed.
1600 routes have been processed.


- It took 9 minutes 49 seconds to finish 1666 routes.

In [8]:
ideal_tolerances.write_csv(PROCESSED_DATA_FOLDER / "ideal_tolerances.csv")

# Investigate anomalies in ideal_tolerances (12h)
- Load ideal_tolerances without performing the search again

In [3]:
ideal_tolerances = pl.read_csv(PROCESSED_DATA_FOLDER / "ideal_tolerances.csv")

In [4]:
hour_order = ["1h", "2h", "3h", "4h", "5h", "6h", "12h"]

fig = px.histogram(
    ideal_tolerances.sort("tolerance", descending=True),
    x="tolerance",
    category_orders={"tolerance": hour_order},
    color="tolerance",
    color_discrete_sequence=px.colors.sequential.Teal,
    labels={"tolerance": "Tolerance window", "count": "Count"},
    title="Distribution of ideal tolerances",
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter, sans-serif", size=13, color="#444"),
    title=dict(
        font=dict(size=16, color="#222"), x=0, xanchor="left", pad=dict(l=0, b=16)
    ),
    xaxis=dict(
        categoryorder="array",
        categoryarray=hour_order,
        showgrid=False,
        showline=True,
        linecolor="#ddd",
        tickfont=dict(size=12),
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#f0f0f0",
        showline=False,
        tickfont=dict(size=12),
    ),
    bargap=0.25,
    margin=dict(l=40, r=20, t=60, b=40),
)

fig.update_traces(marker_line_width=0)

fig.show()

In [5]:
fig = px.histogram(
    ideal_tolerances,
    x="n_rows",
    labels={"n_rows": "Number of Rows", "count": "Count"},
    color_discrete_sequence=["#1FC2B0"],
    title="Distribution of row counts",
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Inter, sans-serif", size=13, color="#444"),
    title=dict(
        font=dict(size=16, color="#222"), x=0, xanchor="left", pad=dict(l=0, b=16)
    ),
    xaxis=dict(
        showgrid=False,
        showline=True,
        linecolor="#ddd",
        tickfont=dict(size=12),
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="#f0f0f0",
        showline=False,
        tickfont=dict(size=12),
    ),
    bargap=0.25,
    margin=dict(l=40, r=20, t=60, b=40),
)

fig.update_traces(marker_line_width=0)

fig.show()

## Routes with 12h tolerances
- By the previous stops selection algorithm, there shouldn't be any stop pairs that require 12h tolerance ranges

In [16]:
ideal_tolerances.filter(pl.col("tolerance") == '12h').select(pl.col("n_rows").sum().alias("total_rows_matched_for_12h"))

total_rows_matched_for_12h
i64
0


- All the routes with 12h tolerance have **zero** rows matched, suggesting that rather than the routes need 12h tolerance, there are data issues present for these routes.

In [7]:
abnormal_routes = ideal_tolerances.filter(pl.col("tolerance") == '12h').select(['route','depart_id','dest_id'])
print(f"There are {len(abnormal_routes)} abnormal routes with 12h as their tolerances.")
print("First investigate")
display(abnormal_routes.filter(pl.col("route") == "183701"))

There are 14 abnormal routes with 12h as their tolerances.
First investigate


route,depart_id,dest_id
str,i64,i64
"""183701""",268417,311648


In [9]:
abnormal_df = df_train.filter(pl.col("Route") == "183701",
                              pl.col("StopID").is_in([268417,311648])).collect()

In [11]:
abnormal_df.group_by("Stop").len()

Stop,len
str,u32
"""永康轉運站""",670
"""朝馬站(國光)""",1767


In [17]:
d = date(2025, 11, 6)
abnormal_df.filter(pl.col("Time").is_between(d, datetime.combine(d, time(23, 59, 59))))

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-1167""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-11-06 14:07:42,"""Thu"""
"""183701""","""KKA-1167""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-11-06 14:10:19,"""Thu"""
"""183701""","""KKA-2611""","""去程""","""永康轉運站""",311648,5,"""進站""",2025-11-06 18:20:25,"""Thu"""
"""183701""","""KKA-2611""","""去程""","""永康轉運站""",311648,5,"""離站""",2025-11-06 18:23:41,"""Thu"""


In [18]:
d = date(2025, 3, 1)
abnormal_df.filter(pl.col("Time").is_between(d, datetime.combine(d, time(23, 59, 59))))

Route,Plate,Direction,Stop,StopID,StopSeq,Event,Time,Day_of_week
str,str,cat,str,i32,i32,cat,datetime[μs],cat
"""183701""","""KKA-1125""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-01 10:21:23,"""Sat"""
"""183701""","""KKA-1125""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-01 10:22:50,"""Sat"""
"""183701""","""KKA-1107""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-01 12:22:06,"""Sat"""
"""183701""","""KKA-1107""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-01 12:25:05,"""Sat"""
"""183701""","""KKA-1103""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-01 16:21:21,"""Sat"""
"""183701""","""KKA-1103""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-01 16:23:27,"""Sat"""
"""183701""","""KKA-1109""","""去程""","""朝馬站(國光)""",268417,4,"""進站""",2025-03-01 20:22:35,"""Sat"""
"""183701""","""KKA-1109""","""去程""","""朝馬站(國光)""",268417,4,"""離站""",2025-03-01 20:32:28,"""Sat"""


## Finding
- For unknown reasons, on any single days, 